# Text to speech (TTS)
- TTS is one of the coolest subtasks in the machine learning and AI space, and one of the fastest growing fields (audio dev) in general today! In my experience, aside from LLM models, I see many TTS (and other audio-task related models) going obsolete and replacing each other by iterating into better models, at an extremely fast pace! It's truly extremely exciting.

## How does the TTS architecture work?
Here is a very high-level outline of how TTS works:

### The "frontend"
  - Text-processing 
    - Normalization (expanding abbreviation, handling special characters, converting numbers): to clean the text
  - Tokenization/phonemization: Processed text needs to get converted into phonemes, which are the distinct   sounds that make up words. Very important step when it comes to accurately capturing how the text should sound when spoken out loud.
    - Example: The word "cat" is represented by the phonemes /k/, /æ/, /t/.
  - Prosody Modeling: Helps make the speech sound more normal and less robotic. This step essentially takes away the feeling that the audio was created by a TTS service (for example, using the speaker option in Google Translate always creates audio that very obviously sounds like it was made using a TTS service). Adds rhythm, stress, intonation, to generated audio to make it sound close to human speech.
### The "backend"
- Speech Synthesis: The system generates the audio waveforms from the processed text, creating the spoken voice that users hear. This is the final output.

## Delving deep
- Now we're going to go through each step of the text-to-speech process in great depth.

### Text preprocessing
- This is a relatively simple step. You essentially process the text into a version that makes transforms it from original raw input text into a format that is suitable for speech suitable.
- Below is an example of how to normalize text, including some of the most common techniques:
    - Abbreviation expansion (e.g., "Dr." → "doctor")
    - Number conversion (e.g., "50.00" → "fifty")
    - Special characters handling (e.g., "$50.00" → "fifty dollars")

- Here is an example of a text-preprocessing function:
```python
def preprocess_for_tts(text):
    # Basic abbreviations dictionary
    abbreviations = {
        'dr.': 'doctor',
    }
   
    # Convert text to lowercase for easier matching
    words = text.lower().split()
    processed_words = []
   
    for word in words:
        # Handle dollar amounts
        if word.startswith('$'):
            number = float(word.replace('$', ''))
            processed_words.append(num2words(number) + " dollars")
        # Expand abbreviations
        elif word in abbreviations:
            processed_words.append(abbreviations[word])
        # Convert plain numbers
        elif word.replace('.', '').isdigit():
            processed_words.append(num2words(float(word)))
        else:
            processed_words.append(word)
   
    return ' '.join(processed_words)

# Example
text = "Dr. Smith paid $50.00"
processed_text = preprocess_for_tts(text)

print(f"Processed: {processed_text}")
```

### Phonetic Conversion
- Phonemes are the smallest unit of sound in speech, and therefore this is an important step when it comes to trying to achieve accurate pronunciation. It allows the program to map written words onto their corresponding spoken sounds.
- We can achieve and demonstrate basic phonetic conversion using the phonemizer library. However, it is **extremely important to note** that production TTS systems (e.g. ElevenLabs, Tacotron, StyleTTS2, Amazon Polly, FastSpeech2, etc) and other TTS systems (Kokoro-82m, Dia-1.6b) tend to use more complex transformer-based grapheme-to-phoneme (G2P, remember this abbreviation!) models that consider context and language-specific rules, giving their outputs more nuance and accuracy/ability.
  - Graphemes are the smallest units of written language that represent a sound (for example, letters or letter groups in English like "c", "sh", or "ough"). These small individual sounds (that still exist as English romanized letters (or whatever the script is for the language you're working with)), get converted to individual phonemes.
- Here's a code example using the basic phonemizer library:

In [ ]:
import phonemizer
# Phonetic transcription of the text
text = "Hello, world!"
phonemes = phonemizer.phonemize(text)

print("Phonetic Transcription:", phonemes)

We will go over how these sophisticated G2P models work in-depth in a separate notebook in this section of the repo.
# THIS IS definitely worth looking into: https://huggingface.co/blog/hexgrad/g2p


### Prosody Modeling
- Prosody was added separately to make speech sound more natural by adjusting pinch, stress and intonation. These are the elements that are so core and important to producting audio output that sounds like actual human spoken voice. However, modern neural TTS engines are much more advanced, and use an end-to-end method. They learn to handle phonetic conversion AND prosody modeling internally during training. Instead of relying on separate rules for phoneme conversion and prosody, these models learn to generate natural-sounding speech directly from text.
- We will now go over how prosody was added to the speech.
- In traditional TTS systems, prosody modeling was rule-based. After the G2P step, the phoneme sequence was just flat. If you synthesized it into speech directly as it was, it would be monotone and robotic, just like Google Translate TTS. So prosody would add some life to it by marking syllables that are stressed, decide durations for each phoneme, and insert pauses for commands and periods, and adding pitch contour (the variation//graph with rise and fall in pitch as a speaker talks, so something like "At this part of the word, pitch is 180 Hz, then it drops to 150 Hz, then rises to 210 Hz". The pitch, also known as fundamental frequency, is usally applied (and measured if voice cloning or training using a voice dataset) over a time interval like every 10 ms, and then applied to the synthesized speech output by adding different levels of pitch to different parts of the speech using the rules set). This would be done using a prosody module with EXPLICIT rules. Engineers literally hard-coded "intonation" and "rhythm" tweaks based on text patterns, like:

- If token is a comma → insert 200 ms pause.
- If token is a question mark → raise pitch 30 Hz at the end.
- If word is in ALL CAPS → increase pitch and amplitude. 
- If number appears → slow down by 20%.

This is why old screen readers or GPS voices sounded very formulaic. They were just applying deterministic if/else rules to punctuation, capitalization, and part-of-speech tags.

Later, people started using statistical models instead, to predict prosody from linguistic features.
The features used for these models often included stuff like part of speech, position in current phrase, syllable stress, the phonemes surrounding it. It would predict labels like $F_0$ (pitch countour, the pitch ever time interval spoken about above), duration per phoneme and energy (amplitude/intensity of the sound). But, these models still relied on manually engineered feature extraction and were trained on small datasets. What this means is, a human would extract the aforementioned features from a small dataset, and feed them into the statistical models, which would then predict the aforementioned labels.

But even here there are issues. The model is limited as it can only learn the features that humans could think of. Anything the human engineer skips forgets or whatever when hand-picking their features for model training, such as emotion, emphasis or context (very important! and you'll see this is a precursor to something specific we will need later to improve our tts abilities (yes, attention is all you need)). Additionally, the types of models used for these mini-tasks (CART trees, HMMs, etc) could only learn simple linear relationships between features and outputs. But real prosody patterns are highly nonlinear, as a question’s pitch contour can vary wildly based on emotion, pacing, speaker style. 
Also the small datasets were a problem, as with a few hours of speech, even good features can’t cover every possible word, phrasing, or intonation pattern, and this is why the system would fall back to overly generic rules, which sound robotic and repetitive.

Also, outputs were stitched back then. After G2P, each phoneme would be used to look up its corresponding sound/audio, and all the audios of all phoenemes would be combined together, and every phoneme added to the sequence would have its pitch and other attributes set depending on the previous phoneme added to the sequence.  That’s why GPS voices sounded choppy; the prosody model was just telling the audio stitcher, "raise pitch here, slow down here," but the source clips weren’t recorded in that exact style.

This is called **concatenative synthesis**.

What came next (many years later), was a neural-based method that used neural networks to create prosody embeddings which would then be turned into mel spectograms (we are going to go over this very soon don't worry).
To fully understand how these methods (and the modern TTS methods that we cover later) work, we first need to understand how sound works, and then build our understanding to mel spectograms so we can apply that knowledge to the TTS pipeline, to make the TTS pipeline fully make sense!
> Note: one of the biggest observations I made while studying material to create this notebook is that many articles I found online that explained how TTS works (as in the entire pipeline) failed to provide an intuitive explanation of how sound works and how that knowledge/methodology relates and applies to the actual TTS pipeline, new or old. I am no physicist, but I will give it my best shot to explain and coherently go over how sound works in this piece below.

### Understanding sound
- Sound is caused by variations in air pressure over time. When a source disturbs air particles, they oscillate back and forth, creating regions of compression (higher air pressure) and rarefaction (lower air pressure). These pressure differences cause nearby particles to oscillate as well, allowing the disturbance to propagate as a longitudinal wave. Because the wave’s energy spreads in all directions and some is lost to absorption, the intensity (difference in air pressure and general movement of particles (displacement from their equilibrium/normal movement)) decreases as the distance from the source increases. 
- This therefore means the sound itself gets weaker as you get further from the source of the disturbance to the air particles. Anyone standing in the nearby space has eardrums which then perceive these air pressure differences as sound.
- So if the particles are stationary (particles are actually never stationary due to thermal motion (and Brownian motion in fluids), but for this example lets assume they are capable of completely stationary behaviour), then no sound will be perceived by nearby humans.
- To make this understanding concrete, here is a graph that shows what a plot of air pressure over time looks like for a silent room vs a room with a sound:

![Air Pressure variation in rooms of varying sound](../media/airpressure.png)

- And to link that visualization to the longitudinal wave that we spoke about earlier to really clarify what the different types of waves really mean and show:

![Longitudinal and transverse wave](../media/longvstrans.jpg)



Now, we will go over how prosody is added to speech in modern TTS; an approach where instead of phoneme conversion and prosody modeling take place at different steps, they generate both directly by combining the two.